This notebook evaluates the sentence classifier on the hold-out test set.

In [1]:
from dap_job_quality.pipeline.find_job_quality import JobQuality, split_into_chunks

from datasets import Dataset
import pandas as pd
import time

job_quality = JobQuality()
job_quality.load()

2024-08-29 16:40:06,466 - datasets - INFO - PyTorch version 2.1.2 available.


/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-08-29 16:40:09,806 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2024-08-29 16:40:09,972 - root - INFO - Loading models and variables
2024-08-29 16:40:10,122 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2024-08-29 16:40:10,321 - root - INFO - Downloading the model...
2024-08-29 16:40:58,029 - root - INFO - Loading the model and tokenizer...
2024-08-29 16:40:58,433 - aiobotocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


2024-08-29 16:40:58,912 - root - INFO - Calculating embeddings for 131 target phrases ...


Batches: 100%|██████████| 5/5 [00:00<00:00,  7.14it/s]


In [2]:
test_df = pd.read_parquet('s3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/test_df_20240725.parquet')
test_df.head()

,id,sentence,label
0,45372524,HRC Recruitment acts both as an employment bus...,0
1,44720449,Hays Specialist Recruitment Limited acts as an...,0
2,47775626,ASC Connections Ltd acts as an employment busi...,0
3,45572929,Required Skills Good work ethic.,0
4,48251662,"Good interpersonal, verbal and written communi...",0


In [4]:
# Make sure there are no super super long sentences
test_df["chunks"] = test_df["sentence"].apply(
            lambda x: split_into_chunks(x) if len(x.split()) >= 25 else [x]
        )
test_df = test_df.explode("chunks").reset_index(drop=True)
test_df = test_df.drop(columns=["sentence"])
test_df = test_df.rename(columns={"chunks": "sentence"})

dataset = Dataset.from_pandas(test_df)

In [5]:
start_time = time.time()
predictions = job_quality.job_quality_classifier(
            dataset["sentence"], batch_size=job_quality.batch_size
        )

elapsed_time = time.time() - start_time
print(f"Time taken: {elapsed_time:.2f} seconds")

labels = []
pred_scores = []
for pred in predictions:
    labels.append(pred["label"])
    pred_scores.append(pred["score"])

test_df["job_quality_label"] = labels
test_df["job_quality_prob"] = pred_scores

job_quality_df = test_df[
            (
                (test_df["job_quality_label"] == "LABEL_1")
                & (test_df["job_quality_prob"] >= job_quality.JQ_THRESHOLD)
            )
        ]


Time taken: 15.11 seconds


In [8]:
test_df

,id,label,sentence,job_quality_label,job_quality_prob
0,45372524,0,HRC Recruitment acts both as an employment bus...,LABEL_0,0.782084
1,44720449,0,Hays Specialist Recruitment Limited acts as an...,LABEL_0,0.740186
2,47775626,0,ASC Connections Ltd acts as an employment busi...,LABEL_0,0.626871
3,45572929,0,Required Skills Good work ethic.,LABEL_0,0.686511
4,48251662,0,"Good interpersonal, verbal and written communi...",LABEL_0,0.806947
...,...,...,...,...,...
439,47866099,1,Evening shift between 2.30pm and 12.30 am over...,LABEL_1,0.815262
440,42758914,1,We have different shifts options available to ...,LABEL_1,0.839869
441,48133784,1,Work visa sponsorship available (if required),LABEL_1,0.772748
442,44017518,1,We reimburse the Tier 2 visa application fee i...,LABEL_1,0.689578


In [11]:
from sklearn.metrics import confusion_matrix, classification_report

# Rename 'LABEL_0' to 0 and 'LABEL_1' to 1 in the 'job_quality_label' column
test_df['job_quality_label'] = test_df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})

conf_matrix = confusion_matrix(test_df['label'], test_df['job_quality_label'])
class_report = classification_report(test_df['label'], test_df['job_quality_label'], output_dict=True)

conf_matrix, class_report

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_91768/3652227693.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_df['job_quality_label'] = test_df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})


(array([[150,  20],
        [ 55, 219]]),
 {'0': {'precision': 0.7317073170731707,
   'recall': 0.8823529411764706,
   'f1-score': 0.8,
   'support': 170.0},
  '1': {'precision': 0.9163179916317992,
   'recall': 0.7992700729927007,
   'f1-score': 0.8538011695906432,
   'support': 274.0},
  'accuracy': 0.831081081081081,
  'macro avg': {'precision': 0.8240126543524849,
   'recall': 0.8408115070845856,
   'f1-score': 0.8269005847953217,
   'support': 444.0},
  'weighted avg': {'precision': 0.8456337243458377,
   'recall': 0.831081081081081,
   'f1-score': 0.8332016226753068,
   'support': 444.0}})